<a href="https://colab.research.google.com/github/kapileswaraputejasri/25B21CS063_DriveReady_opc_2026/blob/main/agenticAIday3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -q langchain langchain-groq

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("groq_api")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! 😊 Yes, we offer eggless versions of all our cakes – plum, chocolate, and cupcakes. Let me know which one you’d like, and I’ll help you with the details!
Your name is Ravi.


In [ ]:
history


[('human', "Hi, I'm Ravi. Do you make eggless cakes?"),
 ('ai',
  'Hi Ravi! 😊 Yes, we offer eggless versions of all our cakes – plum, chocolate, and cupcakes. Let me know which one you’d like, and I’ll help you with the details!'),
 ('human', "What's my name?"),
 ('ai', 'Your name is Ravi.')]

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving test.csv to test.csv


In [ ]:
import csv

def run_tests(path="/content/test.csv"):
    with open(path, mode="r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()
        passed += ok

        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])

    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

In [ ]:
run_tests()

PASS | How much is a plum cake? → A plum cake costs ₹500. Let me know if you’d like to add anything else
FAIL | Do you make eggless cakes? → Yes! All our cakes—including plum, chocolate and cupcakes—are availabl
FAIL | Can I get a cake today evening? → I’ll check with the owner and get back to you shortly!
PASS | Do you deliver to Kakinada? → I'll check with the owner.
PASS | What is the price of 6 cupcakes? → Sure! Six cupcakes would cost ₹480. Let me know if you’d like to add a

3/5 passed (60%)


In [ ]:
from langchain.agents import create_agent

agent_order = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result_order = agent_order.invoke({
    "messages": [
        {"role": "user", "content": "Do you make eggless cakes?"}
    ]
})

print(result_order["messages"][-1].content)

Yes! All our cakes – plum, chocolate and cupcakes – are available in egg‑less versions. Let us know which one you’d like!
